In [3]:
# ================================
# Setup Cell (Robust S&P 500 Universe)
# ================================

import os
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

# ---- Project Structure ----
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
UNIVERSE_DIR = DATA_DIR / "universe"
RAW_DIR = DATA_DIR / "raw"

UNIVERSE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ---- Universe Config ----
UNIVERSE_NAME = "sp500"
TICKER_FILE = UNIVERSE_DIR / f"{UNIVERSE_NAME}.csv"

# ---- Create universe if missing ----
if not TICKER_FILE.exists():
    print("Creating S&P 500 universe...")

    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find("table", {"id": "constituents"})

    sp500 = pd.read_html(str(table))[0]

    tickers = (
        sp500["Symbol"]
        .str.replace(".", "-", regex=False)  # BRK.B → BRK-B
        .str.upper()
        .str.strip()
        .tolist()
    )

    df = pd.DataFrame({"ticker": tickers})
    df.to_csv(TICKER_FILE, index=False)

    print(f"Saved {len(tickers)} tickers to {TICKER_FILE}")

else:
    print("Ticker universe already exists. Using cached file.")

# ---- Load tickers ----
tickers = (
    pd.read_csv(TICKER_FILE)["ticker"]
    .dropna()
    .str.upper()
    .str.strip()
    .unique()
    .tolist()
)

print(f"\nLoaded {len(tickers)} tickers")
print("Sample:", tickers[:10])

Creating S&P 500 universe...
Saved 503 tickers to /Users/neilyejjey/stock_signal_engine_v1/data/universe/sp500.csv

Loaded 503 tickers
Sample: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


/var/folders/l3/hj4t10p12gs0m_nkjpjr9kqr0000gp/T/ipykernel_42944/1293268153.py:37: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500 = pd.read_html(str(table))[0]


In [4]:
tickers

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE',
 'AEP',
 'AXP',
 'AIG',
 'AMT',
 'AWK',
 'AMP',
 'AME',
 'AMGN',
 'APH',
 'ADI',
 'AON',
 'APA',
 'APO',
 'AAPL',
 'AMAT',
 'APP',
 'APTV',
 'ACGL',
 'ADM',
 'ARES',
 'ANET',
 'AJG',
 'AIZ',
 'T',
 'ATO',
 'ADSK',
 'ADP',
 'AZO',
 'AVB',
 'AVY',
 'AXON',
 'BKR',
 'BALL',
 'BAC',
 'BAX',
 'BDX',
 'BRK-B',
 'BBY',
 'TECH',
 'BIIB',
 'BLK',
 'BX',
 'XYZ',
 'BK',
 'BA',
 'BKNG',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BRO',
 'BF-B',
 'BLDR',
 'BG',
 'BXP',
 'CHRW',
 'CDNS',
 'CPT',
 'CPB',
 'COF',
 'CAH',
 'CCL',
 'CARR',
 'CVNA',
 'CASY',
 'CAT',
 'CBOE',
 'CBRE',
 'CDW',
 'COR',
 'CNC',
 'CNP',
 'CF',
 'CRL',
 'SCHW',
 'CHTR',
 'CVX',
 'CMG',
 'CB',
 'CHD',
 'CIEN',
 'CI',
 'CINF',
 'CTAS',
 'CSCO',
 'C',
 'CFG',
 'CLX',
 'CME',
 'CMS',
 'KO',
 'CTSH',
 'COHR',
 'COIN',
 'CL',
 'CMCSA',
 '

In [5]:
# #in case modules not installed
# !pip install --upgrade pip setuptools wheel
# !pip install yfinance

In [6]:
# =========================================
# Download Price Data
# =========================================

import time
import yfinance as yf
import pandas as pd

# ---- Config ----
START_DATE = "2010-01-01"
END_DATE = "2026-01-01"   # exclusive in yfinance
SLEEP_SECONDS = 0.25      # tiny pause to be polite / reduce hiccups
PRICE_FILE = RAW_DIR / "prices.parquet"
FAILED_FILE = RAW_DIR / "failed_tickers.csv"

all_prices = []
failed_tickers = []

for i, ticker in enumerate(tickers, start=1):
    try:
        print(f"[{i}/{len(tickers)}] Downloading {ticker}...")

        df = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=False,   # keep raw OHLC + Adj Close
            progress=False,
            threads=False
        )

        if df.empty:
            print(f"  -> No data returned for {ticker}")
            failed_tickers.append(ticker)
            continue

        # Flatten columns just in case
        df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

        # Standardize column names
        df = df.reset_index().rename(columns={
            "Date": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adj_close",
            "Volume": "volume"
        })

        # Add ticker
        df["ticker"] = ticker

        # Keep only the columns we care about
        expected_cols = ["date", "ticker", "open", "high", "low", "close", "adj_close", "volume"]
        df = df[[col for col in expected_cols if col in df.columns]]

        # Basic cleanup
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values(["ticker", "date"]).drop_duplicates(subset=["ticker", "date"])

        all_prices.append(df)

        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        print(f"  -> Failed for {ticker}: {e}")
        failed_tickers.append(ticker)

# ---- Combine all successful downloads ----
if not all_prices:
    raise ValueError("No price data was downloaded. Check tickers and internet/API availability.")

prices = pd.concat(all_prices, ignore_index=True)
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)

# ---- Save outputs ----
prices.to_parquet(PRICE_FILE, index=False)

failed_df = pd.DataFrame({"ticker": failed_tickers})
failed_df.to_csv(FAILED_FILE, index=False)

print("\nDownload complete.")
print(f"Saved prices to: {PRICE_FILE}")
print(f"Saved failed tickers to: {FAILED_FILE}")
print(f"Price dataset shape: {prices.shape}")
print(f"Unique tickers downloaded: {prices['ticker'].nunique()}")
print(f"Failed tickers: {len(failed_tickers)}")

[1/503] Downloading MMM...
[2/503] Downloading AOS...
[3/503] Downloading ABT...
[4/503] Downloading ABBV...
[5/503] Downloading ACN...
[6/503] Downloading ADBE...
[7/503] Downloading AMD...
[8/503] Downloading AES...
[9/503] Downloading AFL...
[10/503] Downloading A...
[11/503] Downloading APD...
[12/503] Downloading ABNB...
[13/503] Downloading AKAM...
[14/503] Downloading ALB...
[15/503] Downloading ARE...
[16/503] Downloading ALGN...
[17/503] Downloading ALLE...
[18/503] Downloading LNT...
[19/503] Downloading ALL...
[20/503] Downloading GOOGL...
[21/503] Downloading GOOG...
[22/503] Downloading MO...
[23/503] Downloading AMZN...
[24/503] Downloading AMCR...
[25/503] Downloading AEE...
[26/503] Downloading AEP...
[27/503] Downloading AXP...
[28/503] Downloading AIG...
[29/503] Downloading AMT...
[30/503] Downloading AWK...
[31/503] Downloading AMP...
[32/503] Downloading AME...
[33/503] Downloading AMGN...
[34/503] Downloading APH...
[35/503] Downloading ADI...
[36/503] Downloading

In [7]:
prices

,date,ticker,open,high,low,close,adj_close,volume
0,2010-01-04,A,22.453505,22.625179,22.267525,22.389128,19.810984,3815561
1,2010-01-05,A,22.324751,22.331903,22.002861,22.145924,19.595785,4186031
2,2010-01-06,A,22.067240,22.174536,22.002861,22.067240,19.526157,3243779
3,2010-01-07,A,22.017166,22.045780,21.816881,22.038628,19.500841,3095172
4,2010-01-08,A,21.917025,22.067240,21.745352,22.031473,19.494524,3733918
...,...,...,...,...,...,...,...,...
1903823,2025-12-24,ZTS,123.099998,125.690002,123.059998,125.489998,124.956429,2369000
1903824,2025-12-26,ZTS,125.160004,126.320000,124.800003,126.230003,125.693283,3226800
1903825,2025-12-29,ZTS,126.160004,126.849998,125.559998,125.980003,125.444351,4465600
1903826,2025-12-30,ZTS,125.550003,127.599998,125.449997,126.410004,125.872520,3230200
